In [108]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder , OneHotEncoder
from sklearn.model_selection import train_test_split
import pickle

In [109]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [110]:
data = data.drop(['RowNumber','CustomerId', 'Surname' ],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [111]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [112]:
label_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_df = label_encoder_geo.fit_transform(data[['Geography']])

label_encoder_geo.get_feature_names_out(['Geography'])
geo_df_encodede = pd.DataFrame(geo_df.toarray(), columns=label_encoder_geo.get_feature_names_out(['Geography']))

In [113]:
geo_df_encodede

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [114]:
data = pd.concat([data.drop('Geography',axis=1), geo_df_encodede],axis=1)

In [115]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [116]:
X = data.drop('EstimatedSalary',axis=1)
y= data['EstimatedSalary']
X.shape,y.shape

((10000, 12), (10000,))

In [117]:
X_train , X_test , y_train,y_yest = train_test_split(X,y,test_size=0.2, random_state=42)

In [118]:
#Scalers
scaler= StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_train)

In [119]:
## Save encoder and scaler in pickle file
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('label_encoder_geo.pkl', 'wb') as file:
    pickle.dump(label_encoder_geo, file)
with open("stabdarsscaler.pkl", 'wb') as file:
    pickle.dump(scaler, file)

In [120]:
#ANN Regression
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [121]:
model = Sequential([
Dense(64, activation='relu', input_shape= (X_train.shape[1],)),
Dense(32, activation='relu'),
Dense (1)
]

)

/Users/ritesh.rijhwani/Downloads/Practice/Python Series/.venv/lib/python3.14/site-packages/keras/src/layers/core/dense.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [122]:
model.compile(optimizer='adam', loss = "mean_absolute_error",metrics=['mae'] )

In [123]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [124]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime
log_direc = 'log/fit/' + datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
tensorboard_callback = TensorBoard(log_dir=log_direc, histogram_freq=1 )


In [125]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [126]:
history = model.fit(X_train, y_train, validation_data=(X_test, y_yest), epochs=100, callbacks=[early_stopping_callback, tensorboard_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 69536.7578 - mae: 69536.7578 - val_loss: 65595.1328 - val_mae: 65595.1328
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 862us/step - loss: 62436.5742 - mae: 62436.5742 - val_loss: 55999.1289 - val_mae: 55999.1289
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 849us/step - loss: 53508.1875 - mae: 53508.1875 - val_loss: 50872.2695 - val_mae: 50872.2695
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 913us/step - loss: 51787.6406 - mae: 51787.6406 - val_loss: 50622.3867 - val_mae: 50622.3867
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 863us/step - loss: 51653.7773 - mae: 51653.7773 - val_loss: 50674.4727 - val_mae: 50674.4727
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 856us/step - loss: 51626.9922 - mae: 51626.9922 - val_loss: 50717.4883 - val_mae: 50717.4883
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 861us/step - loss: 51571.0469 - mae: 51571.0469 - val_loss: 50662.7930 - val_mae: 50662.7930
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s

In [127]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [128]:
%tensorboard --logdir log/fit/

Reusing TensorBoard on port 6007 (pid 1469), started 0:00:52 ago. (Use '!kill 1469' to kill it.)

In [130]:
test_loss, test_mae = model.evaluate(X_test, y_yest)
print(test_loss, test_mae)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 503us/step - loss: 50517.1680 - mae: 50517.1680
50517.16796875 50517.16796875


In [132]:
model.save('regression_model.h5')